In [2]:
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import mediapipe as mp

In [3]:
df = pd.read_csv('data.csv')
df_balance = df[df['part'].isin(['left hip', 'right hip', 'left shoulder', 'right shoulder'])]
df_balance.to_csv('balance_data.csv', index=False)
df_balance

,Unnamed: 0,frame,part,x,y,z,visibility,presence
1,1,0,left shoulder,0.178230,-0.472519,-0.148855,0.999990,0.999998
2,2,0,right shoulder,-0.118589,-0.528862,-0.055351,0.999972,0.999988
7,7,0,left hip,0.088616,0.002912,0.009693,0.999986,0.999999
8,8,0,right hip,-0.085932,-0.005783,-0.008043,0.999981,0.999999
14,14,1,left shoulder,0.177826,-0.466797,-0.141731,0.999990,0.999999
...,...,...,...,...,...,...,...,...
8302,8302,638,right hip,0.100664,-0.010329,-0.056615,0.999996,1.000000
8308,8308,639,left shoulder,-0.149993,-0.440297,-0.054169,0.999993,0.999999
8309,8309,639,right shoulder,0.161597,-0.513190,-0.061640,0.999998,0.999999
8314,8314,639,left hip,-0.097901,0.005111,0.055603,0.999991,1.000000


In [4]:
df_balance_pivot = df_balance.pivot(index='frame', columns='part', values=['x', 'y', 'z'])
df_balance_pivot.columns = [f"{axis}_{part.replace(' ', '_')}" for axis, part in df_balance_pivot.columns]
df_balance_pivot = df_balance_pivot.reset_index()
#df_balance_pivot

In [5]:
shoulder_width_raw = np.sqrt((df_balance_pivot['x_left_shoulder'] - df_balance_pivot['x_right_shoulder'])**2 + (df_balance_pivot['y_left_shoulder'] - df_balance_pivot['y_right_shoulder'])**2 + (df_balance_pivot['z_left_shoulder'] - df_balance_pivot['z_right_shoulder'])**2)
shoulder_width = shoulder_width_raw.mean()
print(shoulder_width)

0.29855926607476546


In [6]:
df_balance_pivot['com_x'] = (df_balance_pivot['x_left_hip'] + df_balance_pivot['x_right_hip'] ) / 2
df_balance_pivot['com_y'] = (df_balance_pivot['y_left_hip'] + df_balance_pivot['y_right_hip'] ) / 2
df_balance_pivot['com_z'] = (df_balance_pivot['z_left_hip'] + df_balance_pivot['z_right_hip'] ) / 2
df_balance_pivot['std_com_x'] = df_balance_pivot['com_x']/shoulder_width
df_balance_pivot['std_com_y'] = df_balance_pivot['com_y']/shoulder_width
df_balance_pivot['std_com_z'] = df_balance_pivot['com_z']/shoulder_width
window_size = 30
#chuan hoa tu le theo chieu rong cua vai
sway_x = df_balance_pivot['std_com_x'].rolling(window=window_size).std()
sway_y = df_balance_pivot['std_com_y'].rolling(window=window_size).std()
sway_z = df_balance_pivot['std_com_z'].rolling(window=window_size).std()
df_balance_pivot['sway_index'] = np.sqrt(sway_x**2 + sway_y**2 + sway_z**2).fillna(0)
#print(df_balance_pivot[['frame', 'std_com_x', 'std_com_y', 'std_com_z', 'sway_index']])

In [7]:
mid_shoulder_x = (df_balance_pivot['x_left_shoulder'] + df_balance_pivot['x_right_shoulder']) / 2
mid_shoulder_y = (df_balance_pivot['y_left_shoulder'] + df_balance_pivot['y_right_shoulder']) / 2
mid_shoulder_z = (df_balance_pivot['z_left_shoulder'] + df_balance_pivot['z_right_shoulder']) / 2
v_x = df_balance_pivot['com_x'] - mid_shoulder_x
v_y = df_balance_pivot['com_y'] - mid_shoulder_y
v_z = df_balance_pivot['com_z'] - mid_shoulder_z
length_v = np.sqrt(v_x**2 + v_y**2 + v_z**2)
cos = v_y / length_v
cos = np.clip(cos, -1, 1)
df_balance_pivot['Posture_Deviatation'] = np.degrees(np.arccos(cos))/180.0


In [8]:
anchor_x = df_balance_pivot['std_com_x'].iloc[window_size].mean()
anchor_y = df_balance_pivot['std_com_y'].iloc[window_size].mean()
anchor_z = df_balance_pivot['std_com_z'].iloc[window_size].mean()
df_balance_pivot["CoM_Distance"] = np.sqrt((df_balance_pivot['std_com_x'] - anchor_x)**2 + (df_balance_pivot['std_com_y'] - anchor_y)**2 + (df_balance_pivot['std_com_z'] - anchor_z)**2)
df_balance_statistics = df_balance_pivot[['frame', 'sway_index', 'Posture_Deviatation', 'CoM_Distance']]
df_balance_statistics.to_csv('balance_statistics.csv', index=False)

In [9]:
average_sway_index = df_balance_statistics['sway_index'].mean()
average_posture_deviation = df_balance_statistics['Posture_Deviatation'].mean()
average_com_distance = df_balance_statistics['CoM_Distance'].mean()
df_balance_statistics['sway_deviation'] = abs(df_balance_statistics['sway_index'] - average_sway_index)
df_balance_statistics['posture_deviation_deviation'] = abs(df_balance_statistics['Posture_Deviatation'] - average_posture_deviation)
df_balance_statistics['com_distance_deviation'] = abs(df_balance_statistics['CoM_Distance'] - average_com_distance)
sway= df_balance_statistics['sway_deviation'].mean()
posture_deviation = df_balance_statistics['posture_deviation_deviation'].mean()
com_distance_deviation = df_balance_statistics['com_distance_deviation'].mean()
print(df_balance_statistics[['sway_deviation', 'posture_deviation_deviation', 'com_distance_deviation']])

     sway_deviation  posture_deviation_deviation  com_distance_deviation
0          0.000805                     0.006459                0.002607
1          0.000805                     0.008896                0.002660
2          0.000805                     0.010446                0.002690
3          0.000805                     0.013865                0.002696
4          0.000805                     0.013438                0.002707
..              ...                          ...                     ...
635        0.000273                     0.045525                0.003808
636        0.000278                     0.045213                0.003768
637        0.000298                     0.042205                0.003761
638        0.000328                     0.041090                0.003651
639        0.000359                     0.034967                0.003618

[640 rows x 3 columns]


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8956\3455423760.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_balance_statistics['sway_deviation'] = abs(df_balance_statistics['sway_index'] - average_sway_index)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8956\3455423760.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_balance_statistics['posture_deviation_deviation'] = abs(df_balance_statistics['Posture_Deviatation'] - average_posture_deviation)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8956\34

In [10]:
df_balance_statistics.to_csv('balance_statistics.csv', index=False)